In [4]:
import pandas as pd
import numpy as np
import joblib
import os


def load_data_bundle(file_path='./data/processed/final_feature_bundle.joblib'):
    """
    Load toàn bộ tập Train/Test từ file joblib và in ra kích thước kiểm tra.
    """
    if not os.path.exists(file_path):
        print(f"❌ Không tìm thấy file tại: {file_path}")
        return None
    
    # Load bundle
    data = joblib.load(file_path)
    
    X_train = data['X_train']
    X_test = data['X_test']
    y_train = data['y_train']
    y_test = data['y_test']
    
    # Kiểm tra kích thước
    print("--- KIỂM TRA KÍCH THƯỚC DỮ LIỆU ---")
    print(f"Shape X_train: {X_train.shape} | Shape y_train: {y_train.shape}")
    print(f"Shape X_test:  {X_test.shape}  | Shape y_test:  {y_test.shape}")
    
    # Kiểm tra tỷ lệ nợ xấu để đảm bảo Stratified Split hoạt động đúng
    train_default_rate = (y_train.sum() / len(y_train) * 100).round(2)
    test_default_rate = (y_test.sum() / len(y_test) * 100).round(2)
    
    print(f"\n--- TỶ LỆ NỢ XẤU (DEFAULT RATE) ---")
    print(f"Trong tập Train: {train_default_rate}%")
    print(f"Trong tập Test:  {test_default_rate}%")
    
    return X_train, X_test, y_train, y_test


def load_best_model(model_name_keyword='Logistic_Regression', folder_path='./models'):
    """
    2. Load model có AUC cao nhất từ nhật ký benchmark.
    Đã sửa lỗi khớp tên cột 'auc_score'.
    """
    benchmark_file = os.path.join(folder_path, 'model_benchmark_history.csv')
    if not os.path.exists(benchmark_file):
        print(f"❌ Không tìm thấy file: {benchmark_file}")
        return None
    
    # Đọc file history
    history_df = pd.read_csv(benchmark_file)
    
    # Lọc theo keyword (ví dụ 'XGBoost', 'Logistic_Regression')
    filtered_df = history_df[history_df['model_name'].str.contains(model_name_keyword, case=False)]
    
    if filtered_df.empty:
        print(f"⚠️ Không tìm thấy model nào chứa từ khóa: {model_name_keyword}")
        return None
    
    # SỬA LỖI TẠI ĐÂY: Dùng đúng tên cột 'auc_score' từ file CSV của bạn
    best_info = filtered_df.sort_values(by='auc_score', ascending=False).iloc[0]
    
    model_path = os.path.join(folder_path, best_info['file_path'])
    
    try:
        model = joblib.load(model_path)
        print(f"🚀 Đã Load thành công: {best_info['file_path']}")
        print(f"📊 AUC ghi nhận: {best_info['auc_score']:.4f}")
        return model
    except Exception as e:
        print(f"❌ Lỗi khi load file vật lý: {e}")
        return None
    
    
def run_inference(model, X_input, threshold=0.62):
    """
    Dự báo rủi ro dựa trên model và dữ liệu đầu vào đã được xử lý (X_test).
    X_input: DataFrame hoặc Series (1 hoặc nhiều dòng từ X_test)
    """
    # 1. Đảm bảo dữ liệu đầu vào là DataFrame
    if isinstance(X_input, pd.Series):
        X_input = X_input.to_frame().T
    
    # 2. Dự báo xác suất (Probability)
    # Ép kiểu float để tránh lỗi tính toán của Logistic Regression
    probs = model.predict_proba(X_input.astype(float))[:, 1]
    
    # 3. Tạo bảng kết quả
    results = pd.DataFrame({
        'Probability': np.round(probs, 4),
        'Risk_Level': ["🔴 Cao" if p >= threshold else "🟢 Thấp" for p in probs],
        'Decision': ["❌ TỪ CHỐI" if p >= threshold else "✅ DUYỆT" for p in probs]
    })
    
    return results

In [5]:
# Bước 1: Load dữ liệu và Model (Chỉ chạy 1 lần)
X_train, X_test, y_train, y_test = load_data_bundle()
best_model = load_best_model(model_name_keyword="Logistic_Regression")

# Bước 2: Chọn dữ liệu muốn kiểm tra (Ví dụ: 5 dòng đầu tiên của tập Test)
sample_data = X_test.iloc[0:5] 
actual_labels = y_test.iloc[0:5]

# Bước 3: Chạy Inference
inference_results = run_inference(best_model, sample_data)

# Hiển thị kết quả kèm nhãn thực tế để đối soát
inference_results['Actual_Status'] = actual_labels.values
print(inference_results)

--- KIỂM TRA KÍCH THƯỚC DỮ LIỆU ---
Shape X_train: (204277, 22) | Shape y_train: (204277,)
Shape X_test:  (51070, 22)  | Shape y_test:  (51070,)

--- TỶ LỆ NỢ XẤU (DEFAULT RATE) ---
Trong tập Train: 11.61%
Trong tập Test:  11.61%
🚀 Đã Load thành công: Logistic_Regression_AUC07458_20260319_0934.joblib
📊 AUC ghi nhận: 0.7458
   Probability Risk_Level Decision  Actual_Status
0       0.2052     🟢 Thấp  ✅ DUYỆT              0
1       0.1293     🟢 Thấp  ✅ DUYỆT              0
2       0.3572     🟢 Thấp  ✅ DUYỆT              0
3       0.5984     🟢 Thấp  ✅ DUYỆT              0
4       0.4078     🟢 Thấp  ✅ DUYỆT              0
